## Machine learning Notebook - some tests and notes

In [24]:
import pandas as pd 
import numpy as np

In [11]:
# Load data
df = pd.read_csv(r'C:\Users\l.ruizlozano\Documents\Project-Sentinel_data\Rovaniemi\df_Sentinel_ERA5_gold.csv')

In [51]:
df

,year,month,day,DOY,ndsi_median,ndsi_mad,snow_fraction,tp,skt,sd,winter_start,winter,winter_day
0,2018,2,8,39,0.685577,0.062095,0.994837,0.023842,-12.532349,19.487953,2017,2017-2018,2001-02-08
1,2018,2,8,39,0.685577,0.062095,0.994837,0.023842,-12.645630,17.337322,2017,2017-2018,2001-02-08
2,2018,2,8,39,0.685577,0.062095,0.994837,0.022411,-12.532349,19.487953,2017,2017-2018,2001-02-08
3,2018,2,8,39,0.685577,0.062095,0.994837,0.022411,-12.645630,17.337322,2017,2017-2018,2001-02-08
4,2018,2,10,41,0.362368,0.034997,0.029111,0.023365,-3.736938,19.951057,2017,2017-2018,2001-02-10
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1091,2026,4,27,117,-0.169268,0.176689,0.015908,0.072956,5.093140,0.000000,2025,2025-2026,2001-04-27
1092,2026,4,29,119,-0.384942,0.133110,0.005767,0.000000,5.968567,0.000000,2025,2025-2026,2001-04-29
1093,2026,4,29,119,-0.384942,0.133110,0.005767,0.000000,5.867004,0.000000,2025,2025-2026,2001-04-29
1094,2026,4,29,119,-0.384942,0.133110,0.005767,0.000000,5.968567,0.000000,2025,2025-2026,2001-04-29


In [60]:
df.duplicated(subset=["year", "DOY"]).sum()

np.int64(822)

In [63]:
df = df.drop_duplicates(subset=["year", "DOY"])

# Modeling

## Linear Regression: test

In [72]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, root_mean_squared_error
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

In [93]:
X = df[["year","DOY","ndsi_median","tp", "skt", "sd" ]] #features
y = df.snow_fraction #target 

In [94]:
mae, mse, rmse, r2 = [], [], [], []

for elem in list(reversed(df.winter.unique())):
    train_mask = df["winter"] != elem
    test_mask = df["winter"] == elem

    X_train = X.loc[train_mask]
    X_test  = X.loc[test_mask]

    y_train = y.loc[train_mask]
    y_test  = y.loc[test_mask]
    
    model = make_pipeline(
        StandardScaler(),
        LinearRegression()
    )

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    mae.append(mean_absolute_error(y_test, y_pred))
    mse.append(mean_squared_error(y_test, y_pred))
    rmse.append(np.sqrt(mean_squared_error(y_test, y_pred)))
    r2.append(r2_score(y_test, y_pred))

print("---Mean Metrics on all folds---")
print(f"Mean MAE: {np.mean(mae)}")
print(f"Mean MSE: {np.mean(mse)}")
print(f"Mean RMSE: {np.mean(rmse)}")
print(f"Mean R²: {np.mean(r2)}")

---Mean Metrics on all folds---
Mean MAE: 0.16767149720752242
Mean MSE: 0.05613054254948876
Mean RMSE: 0.23000266872820493
Mean R²: 0.469630932137655


# Train_test_split: by winter season

In [114]:
train_mask =  ~df["winter"].isin(["2024-2025", "2025-2026"])
test_mask = df["winter"].isin(["2024-2025", "2025-2026"])

X_train = X.loc[train_mask]
X_test  = X.loc[test_mask]

y_train = y.loc[train_mask]
y_test  = y.loc[test_mask]

In [ ]:
groups = df.loc[train_mask, "winter"]

## Linear Regression

In [ ]:
model = make_pipeline(
        StandardScaler(),
        LinearRegression()
    )

model.fit(X_train, y_train)

LR_y_test_predict = model.predict(X_test)
print("---LR Metrics on test set---")
print(f"MAE: {mean_absolute_error(y_test, LR_y_test_predict)}")
print(f"Mean MSE: {mean_squared_error(y_test, LR_y_test_predict)}")
print(f"Mean RMSE: {np.sqrt(mean_squared_error(y_test, LR_y_test_predict))}")
print(f"Mean R²: {r2_score(y_test, LR_y_test_predict)}")

# Random Forest Regressor

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.ensemble import RandomForestRegressor

param_grid = {"n_estimators": [30,50,100, 200],
    'max_depth': np.arange(1, 20),
    "min_samples_leaf": [1, 2, 5],
    }
groups = df.loc[train_mask, "winter"]


grid = GridSearchCV(RandomForestRegressor(random_state=42, n_jobs=-1), param_grid, cv=LeaveOneGroupOut(), scoring="neg_mean_absolute_error")
grid.fit(X_train, y_train, groups=groups)
print(f"Best estimator: {grid.best_estimator_}")

model = grid.best_estimator_

RF_y_test_predict = model.predict(X_test)

print("---Metrics on test set---")
print(f"MAE: {mean_absolute_error(y_test, y_test_predict)}")
print(f"Mean MSE: {mean_squared_error(y_test, y_test_predict)}")
print(f"Mean RMSE: {np.sqrt(mean_squared_error(y_test, y_test_predict))}")
print(f"Mean R²: {r2_score(y_test, y_test_predict)}")

- Test

In [ ]:
mae, mse, rmse, r2 = [], [], [], []

for elem in list(reversed(df.winter.unique())):
    train_mask = df["winter"] != elem
    test_mask = df["winter"] == elem

    X_train = X.loc[train_mask]
    X_test  = X.loc[test_mask]

    y_train = y.loc[train_mask]
    y_test  = y.loc[test_mask]
    
    model = make_pipeline(
        StandardScaler(),
        RandomForestRegressor(
        max_depth=10,
        random_state=42,
        n_jobs=-1
        )
    )

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    mae.append(mean_absolute_error(y_test, y_pred))
    mse.append(mean_squared_error(y_test, y_pred))
    rmse.append(np.sqrt(mean_squared_error(y_test, y_pred)))
    r2.append(r2_score(y_test, y_pred))

print("---Mean Metrics on all folds---")
print(f"Mean MAE: {np.mean(mae)}")
print(f"Mean MSE: {np.mean(mse)}")
print(f"Mean RMSE: {np.mean(rmse)}")
print(f"Mean R²: {np.mean(r2)}")

# Gradient Boosting Regressor

In [127]:
from sklearn.ensemble import GradientBoostingRegressor

In [ ]:
param_grid = {
    "n_estimators": [100,200,300],
    "learning_rate": [0.03,0.05,0.1],
    "max_depth": np.arange(1, 20),
}

groups = df.loc[train_mask, "winter"]


grid = GridSearchCV(GradientBoostingRegressor(random_state=42), param_grid, cv=LeaveOneGroupOut(), scoring="neg_mean_absolute_error")
grid.fit(X_train, y_train, groups=groups)
print(f"Best estimator: {grid.best_estimator_}")

model = grid.best_estimator_

GB_y_test_predict = model.predict(X_test)

## Comparison Models

In [135]:
models_scores = {
    'Model': ['LinearRegression','RandomForestRegressor', 'GradientBoostingRegressor'],
    'MAE': [mean_absolute_error(y_test, LR_y_test_predict),mean_absolute_error(y_test, RF_y_test_predict),mean_absolute_error(y_test, GB_y_test_predict)],
    'MSE': [mean_squared_error(y_test, LR_y_test_predict),mean_squared_error(y_test, RF_y_test_predict),mean_squared_error(y_test, GB_y_test_predict)],
    'RMSE': [np.sqrt(mean_squared_error(y_test, LR_y_test_predict)), np.sqrt(mean_squared_error(y_test, RF_y_test_predict)), np.sqrt(mean_squared_error(y_test, GB_y_test_predict))],
    'R²': [r2_score(y_test, LR_y_test_predict), r2_score(y_test, RF_y_test_predict), r2_score(y_test, GB_y_test_predict)]
}
display(pd.DataFrame(models_scores))

,Model,MAE,MSE,RMSE,R²
0,LinearRegression,0.299084,0.150602,0.388075,-0.140997
1,RandomForestRegressor,0.106647,0.018888,0.137433,0.856902
2,GradientBoostingRegressor,0.108868,0.020190,0.142090,0.847040


### Mean Snow cover vs Year

- Use TimeSeriesSplit instead of train_test_split as we work on Time series: L'objectif est de prédire le futur à partir du passé, donc on ne mélange jamais les observations. L'idée est que les ensembles d'entraînement grandissent progressivement.

In [32]:
X = snow_area.drop(columns=["snow_fraction"])
y = snow_area["snow_fraction"]
X = np.arange(len(snow_area)).reshape(-1, 1)

In [ ]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, root_mean_squared_error
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

In [47]:
tscv = TimeSeriesSplit(n_splits=3, max_train_size=None, test_size=2)

mae, mse, rmse, r2 = [], [], [], []

for train_index, test_index in tscv.split(X):

    X_train = X[train_index]
    X_test  = X[test_index]
    y_train = y.iloc[train_index]
    y_test  = y.iloc[test_index]

    model = make_pipeline(
        StandardScaler(),
        LinearRegression()
    )

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    mae.append(mean_absolute_error(y_test, y_pred))
    mse.append(mean_squared_error(y_test, y_pred))
    rmse.append(np.sqrt(mean_squared_error(y_test, y_pred)))
    r2.append(r2_score(y_test, y_pred))

print("---Mean Metrics on all folds---")
print(f"Mean MAE: {np.mean(mae)}")
print(f"Mean MSE: {np.mean(mse)}")
print(f"Mean RMSE: {np.mean(rmse)}")
#print(f"Mean R²: {np.mean(r2)}")

---Mean Metrics on all folds---
Mean MAE: 0.13310417927198373
Mean MSE: 0.019161869528184886
Mean RMSE: 0.1362805716895914
